# PixArt-Sigma local latent LoRA smoke test

This thin notebook audits the canonical 260-image assets and calls `scripts/training/train_local_latent_lora.py`. It never contains a second training loop. The current pre-prompt stage is complete when image/latent validation passes and the missing prompt cache is reported as **PENDING**.

Final target: Python 3.11.2, deterministic 50-image subset, rank 8, 10 optimizer updates, adapter save/fresh reload, and one 512 x 512 generation.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'Project-Proposal.md').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from the project checkout.')

ROOT = find_repo_root()
TRAINER = ROOT / 'scripts' / 'training' / 'train_local_latent_lora.py'
LATENT_BUNDLE = ROOT / 'data' / 'archives' / 'clean_latents_512.zip'
IMAGE_ARCHIVE = ROOT / 'data' / 'archives' / 'ink.zip'
PROMPT_CACHE = ROOT / 'data' / 'features' / 'prompt_embeddings_512.pt'
NUM_IMAGES = 50
RANK = 8
MAX_TRAIN_STEPS = 10
SEED = 42
OUTPUT_DIR = ROOT / 'outputs' / 'local_smoke' / f'r{RANK}_n{NUM_IMAGES}'

print('Python:', sys.version.split()[0])
print('Repository:', ROOT)
print('Prompt cache:', PROMPT_CACHE)

## 1. Asset audit

The CLI checks ZIP path safety and integrity, all 260 image-caption pairs, category counts, manifest alignment, fingerprint `b9d3c2d1d404`, latent metadata, `[260, 4, 64, 64]` float16 shape, VAE scaling factor `0.13025`, and finiteness. It does not download a model in validation-only mode.

In [ ]:
audit_command = [
    sys.executable, str(TRAINER),
    '--latent-bundle', str(LATENT_BUNDLE),
    '--image-archive', str(IMAGE_ARCHIVE),
    '--prompt-cache', str(PROMPT_CACHE),
    '--num-images', str(NUM_IMAGES),
    '--seed', str(SEED),
    '--validate-assets-only',
]
subprocess.run(audit_command, cwd=ROOT, check=True)

## 2. Prompt-cache gate

The future `.pt` file must be loadable with `torch.load(..., weights_only=True)` and contain:

```python
{
    'format_version': 1,
    'sample_ids': list[str],
    'prompt_embeds': Tensor[N, 300, 4096],  # float16 CPU
    'attention_masks': Tensor[N, 300],      # bool or int64 CPU
    'max_sequence_length': 300,
    'text_encoder_model': str,
    'manifest_fingerprint': 'b9d3c2d1d404',
}
```

Rows may be unordered and may cover only the selected subset; alignment is always by unique `sample_id`. No zero/random placeholder embedding is accepted as a substitute for the missing production cache.

In [ ]:
PROMPT_READY = PROMPT_CACHE.is_file()
if PROMPT_READY:
    print('READY: prompt cache exists; the next cell can start training.')
else:
    print('PENDING: prompt cache has not been provided.')
    print('Create it according to data/README.md, then rerun this notebook.')

## 3. Train, save, fresh reload, generate

This cell is deliberately gated by the real prompt cache. With the default configuration it runs 10 optimizer updates on the deterministic 50-image subset, saves a rank-8 PEFT adapter, reloads it on a new base transformer, and generates one image using the first selected cached embedding with `guidance_scale=1.0`.

In [ ]:
train_command = [
    sys.executable, str(TRAINER),
    '--latent-bundle', str(LATENT_BUNDLE),
    '--image-archive', str(IMAGE_ARCHIVE),
    '--prompt-cache', str(PROMPT_CACHE),
    '--num-images', str(NUM_IMAGES),
    '--rank', str(RANK),
    '--max-train-steps', str(MAX_TRAIN_STEPS),
    '--seed', str(SEED),
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(map(str, train_command)))
if PROMPT_READY:
    subprocess.run(train_command, cwd=ROOT, check=True)
else:
    print('SKIPPED: waiting for the real prompt embedding cache.')

In [ ]:
from IPython.display import Image as DisplayImage, display

metadata_path = OUTPUT_DIR / 'run_metadata.json'
image_path = OUTPUT_DIR / 'reload_generation.png'
if metadata_path.is_file():
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    print(json.dumps({
        key: metadata[key]
        for key in (
            'status', 'num_images', 'rank', 'optimizer_steps',
            'loss_history', 'train_seconds',
            'peak_allocated_vram_gb', 'generated_image'
        )
    }, indent=2))
    display(DisplayImage(filename=str(image_path)))
else:
    print('No completed training metadata yet; this is expected before the prompt cache arrives.')